# 02. Baseline — Linear Regression «из коробки»

Цель: установить точку отсчёта (baseline). Согласно требованиям CP1, baseline — это **простая модель «из коробки» без feature engineering**.

- Модель: `sklearn.linear_model.LinearRegression`.
- Фичи: только сырые числовые (`CustAccountBalance`, `TransactionTime`) — без любых производных.
- Сплит: стратифицированный по `log1p(target)` (см. `src/preprocessing.py`).
- Метрики: MAE (основная), RMSE, R², MAPE.
- Артефакт: `models/baseline.joblib`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import SEED
from src.preprocessing import load_raw, clean, make_split, TARGET
from src.modeling import set_seed, train_baseline, evaluate, save_model

set_seed(SEED)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "bank_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "baseline.joblib"


## 1. Загрузка и сплит

In [2]:
df = clean(load_raw(RAW_PATH))
train, val, test = make_split(df, test_size=0.2, val_size=0.1, seed=SEED)
print("Sizes:", {"train": len(train), "val": len(val), "test": len(test)})


Sizes: {'train': 733411, 'val': 104773, 'test': 209547}


## 2. Подготовка минимального набора фич

«Из коробки» = только сырые числовые без feature engineering. Берём `CustAccountBalance` и `TransactionTime` (2 признака). Не используем категориальные (требовали бы кодирование) и производные от дат.


In [3]:
FEATURES = ["CustAccountBalance", "TransactionTime"]

def to_xy(df_part):
    return df_part[FEATURES].astype(float), df_part[TARGET].astype(float)

X_train, y_train = to_xy(train)
X_val, y_val = to_xy(val)
X_test, y_test = to_xy(test)

print("X_train shape:", X_train.shape)


X_train shape: (733411, 2)


## 3. Обучение и оценка

In [4]:
model = train_baseline(X_train, y_train)
val_scores = evaluate(model, X_val, y_val)
test_scores = evaluate(model, X_test, y_test)

results = pd.DataFrame({"val": val_scores, "test": test_scores}).round(2)
print(results)


          val     test
MAE   1823.26  1820.12
RMSE  6274.13  6765.23
R2       0.00     0.01
MAPE  2414.68  2114.16


In [5]:
coefs = pd.Series(model.coef_, index=FEATURES, name="coef").round(6)
print("Intercept:", round(float(model.intercept_), 4))
print(coefs)


Intercept: 1367.3884
CustAccountBalance    0.000401
TransactionTime       0.001008
Name: coef, dtype: float64


## 4. Что показывает baseline

- Цифры MAE/RMSE на val/test — это **верхняя граница ошибки**, которую должны побить более сложные модели.
- На сильно скошенном таргете LinearRegression тянет к среднему, а не к медиане → ожидаемо высокий MAE.
- Не делаем никаких выводов о значимости фич по коэффициентам без масштабирования: тут это просто sanity-check.

В `03_experiments.ipynb` сравним этот baseline с:
- Ridge (та же линейная, но с регуляризацией и полным набором фич);
- KNN, RandomForest, LightGBM;
- VotingRegressor-ансамблем;
- лучшей моделью, переобученной на полных данных.


## 5. Сохранение модели

In [6]:
save_model(model, MODEL_PATH)
print("Saved:", MODEL_PATH)


Saved: C:\Users\egorm\PycharmProjects\hsemlcourse-classroom-0b8e58-hseml-group-project-project-template\models\baseline.joblib
